In [1]:
import os
import torch
import librosa
import pandas as pd
from scipy.io import wavfile
from IPython import display as disp
from pathlib import Path
from tqdm import tqdm
from src.modules import ResynthesisWithBottleneckAndTVTimbreModule, ResynthesisWithLogitsAndTVTimbreModule, ResynthesisWithQuantAndTVTimbreModule, SpeakerModule, ResynthesisWithBottleneckModule, ResynthesisWithQuantModule

/home/mtseng/miniconda3/envs/DarkStream/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/mtseng/miniconda3/envs/DarkStream/lib/python3.10/site-packages/speechbrain/utils/torch_audio_backend.py:57: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  available_backends = torchaudio.list_audio_backends()


In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"] = "3" # Set the GPU device
MAX_WAV_VALUE = 32768.0
model_name = "DarkStream-bottleneck-TVT" # Options: DarkStream-bottleneck-TVT, DarkStream-logits-TVT, DarkStream-quant-TVT
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
streaming = True # Set to True if using streaming model
right_context = 4 # Set right context for streaming model, e.g., 4 for 80ms latency

In [3]:
speaker_encoder = SpeakerModule(device="cuda")
use_projector = False
if model_name == "DarkStream-logits-TVT":
    resynthesizer = ResynthesisWithLogitsAndTVTimbreModule.load_from_checkpoint(
        "./ckpts/resynthesis_with_logits_and_tvtimbre.ckpt",
        encoder_path=None,
        map_location="cuda"
    )
    use_projector = True
elif model_name == "DarkStream-bottleneck-TVT":
    # Uncomment the following line to use the KM Resynthesis Module instead
    resynthesizer = ResynthesisWithBottleneckAndTVTimbreModule.load_from_checkpoint(
        "./ckpts/resynthesis_bottleneck_and_tvtimbre.ckpt",
        encoder_path=None,
        decoder_pretrained=None,
        kmeans_path=None,
        map_location="cuda"
    )
elif model_name == "DarkStream-quant-TVT":
    resynthesizer = ResynthesisWithQuantAndTVTimbreModule.load_from_checkpoint(
        "./ckpts/resynthesis_quant_and_tvtimbre.ckpt",
        encoder_path=None,
        map_location="cuda"
    )
elif model_name == "DarkStream-bottleneck":
    resynthesizer = ResynthesisWithBottleneckModule.load_from_checkpoint(
        "./ckpts/resynthesis_with_bottleneck.ckpt",
        encoder_path=None,
        decoder_pretrained=None,
        kmeans_path=None,
        map_location="cuda"
    )
elif model_name == "DarkStream-quant":
    resynthesizer = ResynthesisWithQuantModule.load_from_checkpoint(
        "./ckpts/resynthesis_with_quant.ckpt",
        encoder_path=None,
        map_location="cuda"
    )

resynthesizer = resynthesizer.eval()

2025-09-29 12:02:03 | INFO | speechbrain.utils.fetching | Fetch hyperparams.yaml: Using symlink found at '/home/mtseng/anony_benchmark/models/DarkStreamTVT/pretrained_models/speaker/spkrec-xvect-voxceleb/hyperparams.yaml'
2025-09-29 12:02:03 | INFO | speechbrain.utils.fetching | Fetch custom.py: Fetching from HuggingFace Hub 'speechbrain/spkrec-xvect-voxceleb' if not cached
/home/mtseng/miniconda3/envs/DarkStream/lib/python3.10/site-packages/speechbrain/utils/torch_audio_backend.py:57: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  available_backends = torchaudio.list_audio_backends()
2025-09-29 12:02:03 | DEBUG | speechbr

In [ ]:
data_dir = {
    # source datasets
    'artic': "/data/mtseng/L2Artic/all_data_for_ac_vc",
    'l2artic': "/data/mtseng/L2Artic/all_data_for_ac_vc",
    'vctk': "/data/mtseng/VCTK-Corpus/wav48",
    
    # target datasets
    'emime-eng': "/data/mtseng/EMIME/UEDIN_bilingual_96kHz_data/English_talkers",
    'emime-fin': "/data/mtseng/EMIME/UEDIN_bilingual_96kHz_data/Finnish_talkers",
    'emime-ger': "/data/mtseng/EMIME/UEDIN_bilingual_96kHz_data/German_talkers",
    'emime-man': "/data/mtseng/EMIME/UEDIN_bilingual_96kHz_data/Mandarin_talkers",
} 

metadata_path = "/home/mtseng/anony_benchmark/notebooks/evaluation_data.csv"
metadata_df = pd.read_csv(metadata_path)
out_dir = "/data/mtseng/ICLR2026/results/{}".format(model_name)
if not streaming:
    out_dir += "-ns"
elif right_context > 0:
    out_dir += "-{}ms".format(right_context*20)
print("Output directory:", out_dir)
speaker_embeddings_path = "/home/mtseng/anony_benchmark/notebooks/speaker_embeddings.pkl"
os.makedirs(out_dir, exist_ok=True)

Output directory: /data/mtseng/ICLR2026/results/DarkStream-bottleneck-TVT-80ms


In [5]:
if not os.path.exists(speaker_embeddings_path):
    target_utterances = metadata_df['target_utterance'].unique().tolist()
    speaker_embeddings = []
    for target_utterance in tqdm(target_utterances):
        _dataset = target_utterance.split('/')[1][0]
        if _dataset == 'E':
            _dataset = 'emime-eng'
        elif _dataset == 'F':
            _dataset = 'emime-fin'
        elif _dataset == 'G':
            _dataset = 'emime-ger'
        elif _dataset == 'M':
            _dataset = 'emime-man'
        else:
            raise ValueError("Unknown data dir")

        tgt_audio, _ = librosa.load(os.path.join(data_dir[_dataset], target_utterance), sr=16000)
        tgt_audio = torch.from_numpy(tgt_audio).unsqueeze(0).to(device)

        tgt_embed = speaker_encoder(tgt_audio).squeeze(0).cpu().numpy()
        speaker_embeddings.append({
            'dataset': _dataset,
            'target_utterance': target_utterance,
            'speaker_embedding': tgt_embed
        })
    speaker_embeddings = pd.DataFrame(speaker_embeddings)
    speaker_embeddings.to_pickle(speaker_embeddings_path)
else:
    speaker_embeddings = pd.read_pickle(speaker_embeddings_path)

In [6]:
selected_metadata_df = metadata_df[metadata_df['target_dataset'] == 'emime-eng']
print("Total # utterances to convert: {}".format(len(selected_metadata_df)))

Total # utterances to convert: 6850


In [ ]:
def inference_ns(source_audio_path, tgt_embed, sample_rate=16000):
    src_audio, sr = librosa.load(source_audio_path, sr=sample_rate)
    assert sr == sample_rate
    src_audio = torch.from_numpy(src_audio).unsqueeze(0).unsqueeze(0).to(device)
    resynthesizer.eval()
    with torch.no_grad():
        resyth_audio = resynthesizer(src_audio, tgt_embed)
        resyth_audio = resyth_audio.squeeze()
        resyth_audio = resyth_audio * MAX_WAV_VALUE
        resyth_audio = resyth_audio.detach().cpu().numpy().astype('int16')
    return resyth_audio

def inference(source_audio_path, tgt_embed, model_name, use_projector=False, sample_rate=16000, right_context=4):
    src_audio, sr = librosa.load(source_audio_path, sr=sample_rate)
    assert sr == sample_rate
    src_audio = torch.from_numpy(src_audio).unsqueeze(0).unsqueeze(0).to(device)
    resynthesizer.eval()
    with torch.no_grad():
        original_length = src_audio.shape[-1]
        chunk_size = 3
        frame_samples = 320
        lookahead_size = 7 if "logits" in model_name else right_context

        # The model advances by 'stride' frames each step.
        stride = chunk_size * frame_samples

        # We always need lookahead frames available for the last step.
        lookahead_samples = lookahead_size * frame_samples

        # Number of stride-sized chunks needed to cover original_length
        # (ceil division)
        num_strides = (original_length + stride - 1) // stride

        # Total length required so that:
        # - audio length is a multiple of stride
        # - plus lookahead tail for the final window
        required_length = num_strides * stride + lookahead_samples

        pad_length = max(0, required_length - original_length)

        if pad_length > 0:
            pad = src_audio.new_zeros((*src_audio.shape[:-1], pad_length))
            audio_padded = torch.cat([src_audio, pad], dim=-1)
        else:
            audio_padded = src_audio


        all_audios = []
        all_z_embeds = []
        with resynthesizer.encoder.net.streaming(batch_size=1):
            with resynthesizer.decoder.streaming(batch_size=1):
                for i, offset in enumerate(range(0, original_length, stride)):  # in steps of frame_size, so chunk size is 20ms
                    frame = audio_padded[:, :, offset:offset + stride]
                    if "logits" not in model_name:
                        frame = audio_padded[:, :, offset:offset + stride + lookahead_samples] # add lookahead for quant model
    
                    # print("frame shape", frame.shape)
                    if use_projector:
                        z_embed, _ = resynthesizer.encoder.net(frame)
                    else:
                        z_embed = resynthesizer.encoder.net.encode_to_latent(frame, tr=lookahead_size)
                    if isinstance(z_embed, tuple):
                        z_embed = z_embed[0]
                    if z_embed.shape[-1] == 0:
                        continue
                    if use_projector:
                        z_embed = resynthesizer.projector(z_embed.transpose(1, 2)).transpose(1, 2)

                    if "logits" not in model_name:
                        z_embed = z_embed[:, :, :chunk_size]
                    resynth_audio_chunk, _ = resynthesizer.decoder(z_embed, tgt_embed)

                    all_audios.append(resynth_audio_chunk)
                    all_z_embeds.append(z_embed)
    all_audios = torch.cat(all_audios, dim=-1)
    all_z_embeds = torch.cat(all_z_embeds, dim=-1)

    resyth_audio = all_audios.squeeze().cpu().numpy()
    resyth_audio = resyth_audio * MAX_WAV_VALUE
    resyth_audio = resyth_audio.astype('int16')
    return resyth_audio

In [ ]:
for idx, row in tqdm(selected_metadata_df.iterrows(), total=len(selected_metadata_df)):
    source_dataset = row['source_dataset']
    source_speaker = row['source_speaker']
    source_utterance = row['source_utterance']

    source_audio_path = os.path.join(data_dir[source_dataset], source_utterance)
    assert os.path.exists(source_audio_path), f"File {source_audio_path} does not exist!"
    
    target_dataset = row['target_dataset']
    target_utterance = row['target_utterance']
    target_audio_path = os.path.join(data_dir[target_dataset], target_utterance)
    assert os.path.exists(target_audio_path), f"File {target_audio_path} does not exist!"

    out_wav_path = os.path.join(out_dir, os.path.basename(target_utterance).split('.')[0], "{}_{}".format(source_speaker, os.path.basename(source_utterance)))
    os.makedirs(os.path.dirname(out_wav_path), exist_ok=True)
    if os.path.exists(out_wav_path):
        continue

    tgt_emb = speaker_embeddings[speaker_embeddings['target_utterance'] == target_utterance]['speaker_embedding'].values[0]
    tgt_emb = torch.from_numpy(tgt_emb).unsqueeze(0).to(device)

    if not streaming:
        synth_audio = inference_ns(source_audio_path, tgt_emb)
    else:
        synth_audio = inference(source_audio_path, tgt_emb, model_name=model_name, use_projector=use_projector, right_context=right_context)
    wavfile.write(out_wav_path, 16000, synth_audio)    

53it [00:04, 13.09it/s] [00:00<?, ?it/s]
54it [00:01, 47.27it/s] [00:04<9:03:40,  4.76s/it]
34it [00:00, 41.13it/s] [00:05<5:01:34,  2.64s/it]
58it [00:01, 48.83it/s] [00:06<3:27:39,  1.82s/it]
57it [00:01, 47.79it/s] [00:07<2:59:47,  1.58s/it]
42it [00:00, 44.96it/s] [00:09<2:44:34,  1.44s/it]
36it [00:00, 41.46it/s] [00:10<2:25:16,  1.27s/it]
24it [00:00, 36.12it/s] [00:10<2:10:34,  1.14s/it]
18it [00:00, 32.25it/s] [00:11<1:53:30,  1.00it/s]
68it [00:01, 50.48it/s] [00:12<1:38:16,  1.16it/s]
42it [00:00, 43.76it/s]0 [00:13<1:55:46,  1.02s/it]
48it [00:00, 48.02it/s]0 [00:14<1:54:13,  1.00s/it]
47it [00:00, 47.65it/s]0 [00:15<1:54:29,  1.00s/it]
47it [00:01, 45.76it/s]0 [00:16<1:54:12,  1.00s/it]
42it [00:00, 45.91it/s]0 [00:17<1:55:25,  1.01s/it]
22it [00:00, 35.75it/s]0 [00:18<1:52:24,  1.01it/s]
46it [00:00, 47.61it/s]0 [00:19<1:40:01,  1.14it/s]
39it [00:00, 44.13it/s]0 [00:20<1:43:22,  1.10it/s]
32it [00:00, 39.60it/s]0 [00:21<1:42:57,  1.11it/s]
41it [00:00, 43.97it/s]0 [00:21<

KeyboardInterrupt: 